# 01-01 — Exploration des données GMAO

**Question du projet :** pour chaque équipement du réseau, à une date donnée, **est-il à risque de panne (intervention corrective `CORR`) dans les h prochains jours**, h étant choisi librement (10, 34, 90 jours…), et **dans combien de jours la prochaine panne est-elle attendue** ?

**Objectif de ce notebook :** comprendre l'export GMAO, décider quelles colonnes garder, créer les variables du modèle et choisir celles à conserver avant la modélisation.

| Phase | Contenu |
|---|---|
| 0 | Imports globaux |
| 1 | Configuration globale |
| 2 | Chargement des données |
| 3 | Vue d'ensemble : aperçu, valeurs manquantes, doublons, types de variables |
| 4 | Analyse de la cible |
| 5 | Analyse des colonnes |
| 6 | Feature engineering |
| 7 | Corrélations croisées |
| 8 | Synthèse et décisions |

Les fonctions utilisées sont dans `src/` et les paramètres dans `settings/params.py` : ce notebook ne fait qu'appeler, afficher et interpréter.

## 0. Imports globaux

### Description de l'étape


Toutes les bibliothèques sont importées ici, en une seule fois :
- **données** : `numpy`, `pandas` ;
- **graphiques** : `plotly` (interactifs : on survole pour lire les valeurs), `missingno` (valeurs manquantes) ;
- **statistiques** : `scikit-learn` (information mutuelle).

Le dossier du projet est ajouté au chemin de Python pour pouvoir importer `settings/` et `src/`.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import missingno as msno
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.feature_selection import mutual_info_classif

from settings.params import *
from src.data.make_dataset import load_data
from src.data.preprocess import (uniformiser, supprimer_doublons, convertir_dates, prepare,
                                 nettoyer_texte, est_exclu, groupe_zone)
from src.features.build import (EQ, construire_ensembles, etiquette_horizon, kaplan_meier, proba_panne,
                                delai_median, cramers_v, correlation_cible, paires_redondantes, regle_redondance)
from src.utils.plots import boxplots_par_classe, courbes_survie
from src.utils.logger import logger

### Analyse & Interprétation


L'environnement est prêt si la cellule s'exécute sans erreur. Toutes les fonctions viennent de `src/` et tous les paramètres de `settings/params.py` : le notebook ne contient aucune règle de calcul cachée, et une modification faite dans `src/` profite à tous les notebooks.

## 1. Configuration globale

### Description de l'étape


Options d'affichage, couleurs fixes pour tout le notebook, et rappel des paramètres du projet (lus dans `settings/params.py`).

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

TEMPLATE = "plotly_white"
COUL_OT = {1: "#d1495b", 0: "#9aa5b1"}               # correctif / autre OT
NOMS_OT = {1: "Correctif (CORR)", 0: "Autre OT"}
COUL_Y = {1: "#d1495b", 0: "#2a78d6"}                # panne / pas de panne (dans l'horizon étudié)

print("Fichier de données :", FICHIER_OT, "| trouvé :", FICHIER_OT.exists())
print(f"Période : {DEBUT:%d/%m/%Y} -> {FIN:%d/%m/%Y} | trou d'extraction : {TROU_DEBUT:%d/%m/%Y} -> {TROU_FIN:%d/%m/%Y}")
print(f"Suivi maximal : {H_MAX} jours | horizons étudiés : {HORIZONS}")
print(f"Panne = {TYPE_PANNE}, datée par {DATE_COL} ; OT connu à partir de {SAISIE_COL} (filtre : {FILTRE_SAISIE})")
print("Familles hors périmètre :", EXCLURE_FAMILLES)

### Analyse & Interprétation


Les paramètres affichés guident tout le projet : la panne est un `CORR`, datée par sa **réalisation** ; un OT n'est **connu** qu'à partir de sa **saisie** ; la période va du 1er janvier 2019 au 31 décembre 2025, sans suivi pendant le trou d'extraction de juin-juillet 2025 ; on suit chaque situation au plus 365 jours ; les codes qui ne sont pas des équipements du réseau (localités, unités, groupes d'équipements, véhicules) sont hors périmètre.

## 2. Chargement des données

### Description de l'étape


`load_data()` (fichier `src/data/make_dataset.py`) lit l'export GMAO. Le tableau chargé (`data`) n'est **jamais modifié** : toutes les analyses se font sur une copie (`df`).

In [ ]:
data = load_data()
df = data.copy()
df.head(3)

### Analyse & Interprétation


Le fichier contient **176 047 OT** décrits par **25 colonnes**. Une ligne = un ordre de travail : une intervention sur un équipement, avec son type, ses dates, son coût et la description de l'équipement.

## 3. Vue d'ensemble

### Description de l'étape


Trois questions :
1. **Structure** : combien de lignes, de colonnes, quels types ?
2. **Contenu** : à quoi ressemblent les premières lignes ?
3. **Statistiques** : distributions, valeurs uniques, valeurs les plus fréquentes.

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

### Analyse & Interprétation


**Structure (`info()`) :**
- Les **dates sont lues comme du texte** (`str`) : il faudra les convertir avant tout calcul de mois.
- `codeEquipement` est de type `object` : il mélange quelques nombres et du texte. On le mettra entièrement en texte.
- Les colonnes qui décrivent l'OT lui-même (`codeOT`, `codeEquipement`, `typeIntervention`, `zone`, `fonction`, `dateCreation`, `dateDebutPrevue`) sont **complètes**.

**Statistiques (`describe()`) :**
- **46 564 codes équipement** ; le plus fréquent, `UED/PG` (2 314 OT), est une unité et non un équipement (voir 5.3).
- **15 types d'intervention**, `CORR` étant le plus fréquent (112 834 OT).
- **268 familles**, **154 fonctions**, **40 zones**, **30 états d'OT**.
- `coutMaintenance` : médiane de 23 888 contre une moyenne de 347 903. Les coûts sont **très asymétriques** : quelques interventions très chères tirent la moyenne vers le haut.

### 3.1 Valeurs manquantes

### Description de l'étape


Pour chaque colonne : nombre et pourcentage de valeurs vides. Le graphique `missingno` montre la part de valeurs **présentes** par colonne (une barre pleine = colonne complète).

In [ ]:
manq = df.isna().sum().to_frame("manquantes")
manq["%"] = (manq["manquantes"] / len(df) * 100).round(1)
display(manq[manq["manquantes"] > 0].sort_values("manquantes", ascending=False))

msno.bar(df, figsize=(12, 4), fontsize=9, color="#2a78d6")
plt.title("Part de valeurs présentes par colonne", fontsize=11)
plt.show()

### Analyse & Interprétation


- **Aucune valeur manquante** dans `codeOT`, `codeEquipement`, `typeIntervention`, `zone`, `fonction`, `dateCreation`, `dateDebutPrevue`, `DatePlanifiee` et les coûts.
- `SOURCE` est **entièrement vide** ; `Alimente_par` (91 %), `numeroPlan` (78 %) et `entiteEquipement` (72 %) sont **presque vides** : elles seront difficilement exploitables.
- `Equipement_Parent` manque pour **12 %** des OT, `classeIntervention` pour 7 %.
- `familleEquipement` manque pour **453 OT** (0,3 %), `entiteIntervention` pour 889 OT (0,5 %).

Le choix des colonnes à garder est fait en 3.3 et justifié en 5.10.

### 3.2 Doublons

### Description de l'étape


Deux vérifications :
- **lignes entièrement identiques** ;
- **même numéro d'OT (`codeOT`) sur plusieurs lignes** : un même OT ne doit compter qu'une fois.

Pour les OT en double, on regarde quelles colonnes diffèrent entre les deux lignes.

In [ ]:
print("Lignes entièrement identiques :", df.duplicated().sum())

dup = df[df["codeOT"].duplicated(keep=False)]
print(f"OT en double : {dup['codeOT'].nunique():,} OT, {len(dup):,} lignes")
print("\nValeurs de entiteIntervention dans ces lignes :")
print(dup["entiteIntervention"].value_counts().head(5).to_string())

diff = (dup.groupby("codeOT").nunique() > 1).sum()
print("\nNombre d'OT dont les deux lignes diffèrent, par colonne :")
print(diff[diff > 0].sort_values(ascending=False).to_string())

### Analyse & Interprétation


- **Aucune ligne entièrement identique.**
- **732 OT apparaissent sur deux lignes** (1 488 lignes). Dans presque tous les cas, une des deux lignes porte `DD` dans `entiteIntervention`, l'autre l'unité réelle.
- Les deux lignes diffèrent surtout par `entiteIntervention` ; pour environ 170 OT, l'état, la fonction ou le coût diffèrent aussi. La **date de réalisation n'est différente que pour un seul OT**.

**Décision :** on garde une ligne par OT, celle avec l'unité plutôt que `DD` (`supprimer_doublons`). Comme la date est presque toujours la même, ce choix ne change pas le comptage des pannes : c'est une **précaution**.

### 3.3 Types de variables

### Description de l'étape


Chaque colonne est rangée selon sa **nature** :
- **identifiant** : sert à reconnaître un OT ou un équipement, sans signification en soi ;
- **catégorielle** : du texte qui range l'OT dans une catégorie ;
- **numérique** : un nombre ;
- **date**.

La colonne « gardée » indique si l'on en a besoin pour le projet (définir la panne, la dater, identifier l'équipement, créer une variable, ou nettoyer). La raison des colonnes écartées est donnée en phase 5.9.

In [ ]:
TYPES = {
    "identifiant": ["codeOT", "codeEquipement", "intervention", "Equipement_Parent", "Alimente_par", "numeroPlan"],
    "catégorielle": ["typeIntervention", "familleEquipement", "fonction", "zone", "etatOT", "classeIntervention",
                     "entiteIntervention", "entiteAction", "entiteEquipement"],
    "numérique": ["coutMaintenance", "CoutMainDoeuvre", "CoutMateriel", "CoutService", "CoutMoyen", "SOURCE"],
    "date": ["dateCreation", "dateDebutPrevue", "dateFinPrevue", "DatePlanifiee"],
}
types = pd.DataFrame([{"colonne": c, "type": t, "format pandas": str(df[c].dtype),
                       "valeurs distinctes": df[c].nunique(), "% manquantes": round(df[c].isna().mean() * 100, 1),
                       "gardée": "oui" if c in COLONNES_GARDEES else "non"}
                      for t, cols in TYPES.items() for c in cols])
assert set(types["colonne"]) == set(df.columns)          # aucune colonne oubliée
display(types)
types.groupby(["type", "gardée"]).size().unstack(fill_value=0)

### Analyse & Interprétation


Les 25 colonnes se répartissent en **6 identifiants, 9 catégorielles, 6 numériques et 4 dates**. On en garde **11** :
- **pour définir et dater la panne** : `typeIntervention`, `dateDebutPrevue`, `dateCreation` ;
- **pour identifier l'équipement** : `codeEquipement` ;
- **pour créer des variables** : `familleEquipement`, `fonction`, `zone`, `coutMaintenance`, `Equipement_Parent` ;
- **pour le nettoyage** : `codeOT`, `entiteIntervention` (doublons).

Le format pandas confirme que les **4 dates sont stockées en texte**.

## 4. Analyse de la cible

### Description de l'étape


Au niveau du fichier, la variable d'intérêt est : **l'OT est-il un correctif ?** (`typeIntervention == "CORR"`). On regarde :
- la **répartition** correctifs / autres OT ;
- l'**évolution par mois**, en datant chaque OT par sa date de réalisation (`dateDebutPrevue`).

**Attention :** ce n'est pas encore la cible du modèle. Le modèle prédit, pour un **équipement**, le **délai jusqu'à sa prochaine panne** (phase 6).

In [ ]:
df = uniformiser(df)
df = convertir_dates(df)
df["CORRECT"] = (df["typeIntervention"] == TYPE_PANNE).astype(int)

rep = df["CORRECT"].value_counts().reindex([1, 0]).rename(index=NOMS_OT)
print(rep.to_string(), f"\nPart de correctifs : {df['CORRECT'].mean():.1%}")

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "domain"}, {"type": "xy"}]],
                    subplot_titles=["Répartition", "Nombre d'OT"])
fig.add_trace(go.Pie(labels=rep.index, values=rep.values, hole=0.45,
                     marker_colors=[COUL_OT[1], COUL_OT[0]]), 1, 1)
fig.add_trace(go.Bar(x=rep.index, y=rep.values, marker_color=[COUL_OT[1], COUL_OT[0]],
                     text=rep.values, texttemplate="%{text:,}", showlegend=False), 1, 2)
fig.update_layout(title="Correctifs et autres OT", template=TEMPLATE, height=380)
fig.show()

In [ ]:
mensuel = (df.assign(mois=df[DATE_COL].dt.to_period("M").astype(str))
             .groupby(["mois", "CORRECT"]).size().unstack(fill_value=0).rename(columns=NOMS_OT))
mensuel = mensuel.loc["2019-01":"2026-02"]
fig = px.line(mensuel, x=mensuel.index, y=list(mensuel.columns), template=TEMPLATE,
              color_discrete_map={NOMS_OT[1]: COUL_OT[1], NOMS_OT[0]: COUL_OT[0]},
              title="OT par mois de réalisation (dateDebutPrevue)", labels={"value": "nombre d'OT", "x": "mois"})
fig.update_layout(height=380, legend_title=None)
fig.show()

### Analyse & Interprétation


- **64,1 % des OT sont des correctifs** (112 834 sur 176 047). La maintenance est donc surtout **corrective**, ce qui justifie l'intérêt de prévoir les pannes.
- Le nombre de correctifs par mois **augmente nettement** entre 2019 et 2025 : davantage d'équipements suivis dans la GMAO et une saisie plus complète, surtout dans les régions.
- On voit un **creux en juin et juillet 2025** : ce sont les mois d'extraction incomplète.

**À retenir :** ce 64 % n'est **pas** la cible du modèle. Le modèle prédit, pour un **équipement** à une date donnée, le **délai jusqu'à sa prochaine panne**. La probabilité qu'un équipement tombe en panne dans les 30 jours est bien plus faible (environ 4 %, voir 6.2).

## 5. Analyse des colonnes

### Description de l'étape


On analyse uniquement les colonnes **gardées**. Pour chacune : ce qu'elle contient, ses statistiques, puis son lien avec la cible (correctif ou non). Les colonnes écartées sont listées en 5.9 avec leur raison.

### 5.1 `typeIntervention`

### Description de l'étape

Les différents types d'intervention et leur poids.

In [ ]:
t = df["typeIntervention"].value_counts().rename_axis("type").reset_index(name="nb_OT")
t["%"] = (t["nb_OT"] / len(df) * 100).round(1)
display(t)
fig = px.bar(t, x="type", y="nb_OT", text="nb_OT", template=TEMPLATE, title="Nombre d'OT par type d'intervention",
             color=t["type"].map({TYPE_PANNE: "Correctif", TYPE_PREVENTIF: "Préventif"}).fillna("Autre"),
             color_discrete_map={"Correctif": COUL_OT[1], "Préventif": "#1baf7a", "Autre": COUL_OT[0]})
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_layout(height=400, legend_title=None, yaxis_title="nombre d'OT")
fig.show()

### Analyse & Interprétation


- `CORR` (64 %) et `PREV` (28 %) représentent **92 % des OT**.
- `GARAGE` (5 %) concerne les véhicules, hors périmètre.
- Les 12 autres types pèsent moins de 1 % chacun.

**Décision :** `CORR` = **panne** (la cible) ; `PREV` = **préventif**, utilisé comme variable explicative ; les autres types ne servent qu'à savoir depuis quand un équipement est connu.

### 5.2 Les dates : réalisation et saisie

### Description de l'étape


- `dateDebutPrevue` : date réelle de l'intervention, donc **date de la panne** ;
- `dateCreation` : date de **saisie** dans la GMAO, donc date à partir de laquelle l'OT est **connu**.

On compare les deux pour les correctifs : étendue, retard de saisie, et nombre de correctifs par mois selon chaque date.

In [ ]:
corr = df[df["CORRECT"] == 1].copy()
print(df[[DATE_COL, SAISIE_COL]].agg(["min", "max"]).T)

corr["retard_jours"] = (corr[SAISIE_COL] - corr[DATE_COL]).dt.days
print("\nRetard de saisie des correctifs (jours) :")
print(corr["retard_jours"].describe(percentiles=[.25, .5, .75, .9, .99]).round(0).to_string())
meme = (corr[SAISIE_COL].dt.to_period("M") == corr[DATE_COL].dt.to_period("M")).mean()
print(f"\nSaisis le même mois que la réalisation : {meme:.1%} | saisis avant la réalisation : {(corr['retard_jours'] < 0).mean():.1%}")

r = corr["retard_jours"].clip(-60, 365)
h, b = np.histogram(r, bins=np.arange(-60, 372, 7))
fig = go.Figure(go.Bar(x=b[:-1], y=h, marker_color=COUL_OT[1]))
fig.update_layout(title="Retard de saisie des correctifs (jours, borné à -60 et 365)", template=TEMPLATE,
                  height=350, xaxis_title="jours entre réalisation et saisie", yaxis_title="nombre de correctifs")
fig.show()

In [ ]:
periode = pd.period_range("2019-01", "2026-02", freq="M")
par_debut = corr.groupby(corr[DATE_COL].dt.to_period("M")).size().reindex(periode, fill_value=0)
par_saisie = corr.groupby(corr[SAISIE_COL].dt.to_period("M")).size().reindex(periode, fill_value=0)
fig = go.Figure()
fig.add_trace(go.Scatter(x=periode.astype(str), y=par_saisie.values, name="par date de saisie", line_color="#b9b8b2"))
fig.add_trace(go.Scatter(x=periode.astype(str), y=par_debut.values, name="par date de réalisation", line_color="#2a78d6",
                         line_width=3))
fig.update_layout(title="Correctifs par mois selon la date utilisée", template=TEMPLATE, height=400,
                  yaxis_title="correctifs par mois")
fig.show()
print("Mois avec le plus de correctifs SAISIS :")
print(par_saisie.sort_values(ascending=False).head(4).to_string())

### Analyse & Interprétation


**Étendue :** `dateDebutPrevue` va de 1970 à fin 2026. Il y a **quelques dates aberrantes** (5 OT avant 2017, dont un en 1970) et **467 OT prévus après février 2026** (interventions planifiées). Ils sont **automatiquement écartés** par la période étudiée (2019 à 2025).

**Retard de saisie :**
- médiane : **0 jour**, et **71 %** des correctifs sont saisis le même mois que leur réalisation ;
- **23 %** sont saisis **avant** leur réalisation (OT créés à l'avance) ;
- mais **10 % sont saisis plus de 88 jours après**, et 1 % plus de 4 ans après.

**Les deux courbes :**
- par **date de saisie** (gris), deux pics énormes : **juillet 2024 (10 019)** et **septembre 2024 (6 377)**. Ce sont des **saisies en masse** d'OT anciens ;
- par **date de réalisation** (bleu), la courbe reste régulière.

**Décisions :**
1. Une panne est **datée par `dateDebutPrevue`** : sinon le modèle apprendrait de fausses explosions de pannes.
2. À une date donnée, une partie des pannes récentes n'est pas encore saisie : le modèle ne doit voir que les OT **déjà saisis** (`FILTRE_SAISIE`).
3. Juin et juillet 2025 sont un trou d'extraction : aucune date de coupure et aucun suivi dans ce trou (`TROU_DEBUT`, `TROU_FIN`).

### 5.3 `codeEquipement`

### Description de l'étape


- Combien d'équipements, combien de correctifs par équipement ?
- **Quels équipements ont le plus de correctifs ?**
- Les pannes sont-elles concentrées sur peu d'équipements ? La **courbe de concentration** montre la part des correctifs portée par les X % d'équipements les plus touchés.

In [ ]:
print(f"Équipements distincts : {df['codeEquipement'].nunique():,} | avec au moins un correctif : {corr['codeEquipement'].nunique():,}")
par_eq = corr.groupby("codeEquipement").agg(correctifs=("codeOT", "size"), famille=("familleEquipement", "last"),
                                            zone=("zone", "last")).sort_values("correctifs", ascending=False)
print("\nCorrectifs par équipement (équipements touchés au moins une fois) :")
print(par_eq["correctifs"].describe(percentiles=[.5, .75, .9, .99]).round(1).to_string())
print(f"Touchés une seule fois : {(par_eq['correctifs'] == 1).mean():.0%}")

top = par_eq.head(20).reset_index()
top["périmètre"] = np.where(est_exclu(top["famille"]), "hors périmètre", "équipement du réseau")
fig = px.bar(top, x="correctifs", y="codeEquipement", color="périmètre", orientation="h", template=TEMPLATE,
             hover_data=["famille", "zone"], title="Les 20 équipements avec le plus de correctifs",
             color_discrete_map={"hors périmètre": "#b9b8b2", "équipement du réseau": COUL_OT[1]})
fig.update_layout(height=550, yaxis={"categoryorder": "total ascending"}, legend_title=None)
fig.show()
display(top[["codeEquipement", "correctifs", "famille", "zone", "périmètre"]])

In [ ]:
reel = par_eq[~est_exclu(par_eq["famille"])]["correctifs"].sort_values(ascending=False)
part_eq = np.arange(1, len(reel) + 1) / len(reel) * 100
part_corr = reel.cumsum().values / reel.sum() * 100
idx = np.unique(np.linspace(0, len(reel) - 1, 300).astype(int))
fig = go.Figure(go.Scatter(x=part_eq[idx], y=part_corr[idx], line_color=COUL_OT[1], name="équipements du réseau"))
fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], line=dict(dash="dot", color="#3d3d3a"), name="répartition égale"))
fig.update_layout(title="Concentration des correctifs (équipements du réseau touchés au moins une fois)",
                  xaxis_title="% des équipements, les plus touchés d'abord", yaxis_title="% des correctifs",
                  template=TEMPLATE, height=420)
fig.show()
for p in (1, 10, 20):
    print(f"Les {p} % d'équipements les plus touchés portent {np.interp(p, part_eq, part_corr):.0f} % des correctifs")

### Analyse & Interprétation


- **46 564 codes**, dont **30 139** avec au moins un correctif. **La moitié** de ces équipements n'a eu **qu'un seul** correctif en 7 ans.
- **Le top 20 est presque entièrement hors périmètre** : 18 localités (`UED/PG`, `UED/KM`… avec jusqu'à 2 291 correctifs) et une unité `FAMDEF`. Ce ne sont pas des équipements : ils confirment la décision de les exclure.
- Le seul « équipement du réseau » du top 20 est **`C4`** (famille `OGC`, zone `1000`, 463 correctifs) : **à confirmer avec l'encadrant**.
- **Concentration (équipements du réseau)** : les 10 % les plus touchés portent **46 %** des correctifs, les 20 % les plus touchés **62 %**.

**Conséquence :** les pannes se répètent sur les mêmes équipements. **L'historique de pannes** d'un équipement sera donc un signal important pour le modèle.

### 5.4 `familleEquipement`

### Description de l'étape


- Nombre de familles, valeurs manquantes.
- Pour chaque famille : nombre d'OT, d'équipements, de correctifs et **taux de correctifs**.
- Repérage des familles **hors périmètre** (localités, unités, groupes d'équipements, véhicules).

In [ ]:
fam = df.groupby("familleEquipement").agg(nb_OT=("codeOT", "size"), nb_equipements=("codeEquipement", "nunique"),
                                          nb_correctifs=("CORRECT", "sum"), taux_correctifs=("CORRECT", "mean"))
fam["correctifs_par_equipement"] = fam["nb_correctifs"] / fam["nb_equipements"]
fam["périmètre"] = np.where(est_exclu(fam.index), "hors périmètre", "équipement du réseau")
print(f"Familles : {len(fam)} | OT sans famille : {df['familleEquipement'].isna().sum()}")
display(fam.sort_values("nb_correctifs", ascending=False).head(20).round(2))

print("Familles hors périmètre :")
display(fam[fam["périmètre"] == "hors périmètre"].round(2))

In [ ]:
f = fam[fam["nb_OT"] >= 100].reset_index()
fig = px.scatter(f, x="nb_OT", y="taux_correctifs", size="nb_equipements", color="périmètre", log_x=True,
                 hover_name="familleEquipement", template=TEMPLATE, size_max=45,
                 color_discrete_map={"hors périmètre": "#b9b8b2", "équipement du réseau": COUL_OT[1]},
                 title="Familles : nombre d'OT contre taux de correctifs (taille = nombre d'équipements, familles d'au moins 100 OT)")
fig.update_layout(height=480, yaxis_tickformat=".0%", legend_title=None)
fig.show()

### Analyse & Interprétation


- **268 familles** ; la plus importante, `DEPART_BT`, compte 37 773 OT et 31 655 correctifs.
- Pour la plupart des familles du réseau, le **taux de correctifs dépasse 80 %** (`CABLE_HTA` 97 %, `POSTE_HTA/BT` 86 %). Les familles à faible taux sont celles qui reçoivent surtout du préventif ou d'autres interventions (`IAT` 45 %).

**Familles hors périmètre :**

| Famille | Ce que c'est | Ce que montrent les données |
|---|---|---|
| `LOCALITE` | zones géographiques | 174 codes, **83 correctifs par code** : beaucoup trop pour un équipement |
| `FAMDEF` | unités | 33 codes, 18 correctifs par code |
| `GRPMT` | groupes d'équipements | 11 728 codes, presque aucun correctif (12 % des OT) |
| `V-EXPLO`, `V-CADRE` | véhicules | 1 % de correctifs, zone `GARAGE` |
| `VEH_LEGER`, `VEH_LOURD` | véhicules | 152 OT |

**Décision :** ces codes restent dans les données (leur historique est réel), mais sont **exclus de l'analyse des variables, de l'évaluation du modèle et de la liste de risque**.

### 5.5 `fonction`

### Description de l'étape

La fonction est l'**unité** qui gère l'équipement. Nombre de modalités, unités les plus chargées en correctifs, taux de correctifs.

In [ ]:
fo = df.groupby("fonction").agg(nb_OT=("codeOT", "size"), nb_correctifs=("CORRECT", "sum"),
                                taux_correctifs=("CORRECT", "mean")).sort_values("nb_correctifs", ascending=False)
print(f"Fonctions : {len(fo)} | OT sans fonction : {df['fonction'].isna().sum()}")
print(f"Les 20 premières portent {fo['nb_correctifs'].head(20).sum() / fo['nb_correctifs'].sum():.0%} des correctifs")
t = fo.head(20).reset_index()
fig = px.bar(t, x="nb_correctifs", y="fonction", orientation="h", color="taux_correctifs", template=TEMPLATE,
             color_continuous_scale="Reds", title="Les 20 fonctions avec le plus de correctifs (couleur = taux de correctifs)")
fig.update_layout(height=550, yaxis={"categoryorder": "total ascending"}, coloraxis_colorbar_tickformat=".0%")
fig.show()

### Analyse & Interprétation


- **154 fonctions** (unités qui gèrent les équipements), aucune valeur manquante.
- Les **20 premières portent 82 % des correctifs** : l'activité est concentrée sur quelques unités, surtout à Dakar.
- Les taux de correctifs varient d'une unité à l'autre : la fonction décrit à la fois **le lieu et la façon de travailler** de l'unité, ce qui peut aider le modèle.

### 5.6 `zone`

### Description de l'étape


Zones (après nettoyage des espaces), puis regroupement **Dakar / Régions / Autres** (`groupe_zone`) qui servira à comparer les performances du modèle.

In [ ]:
zo = df.groupby("zone").agg(nb_OT=("codeOT", "size"), nb_correctifs=("CORRECT", "sum"),
                            taux_correctifs=("CORRECT", "mean")).sort_values("nb_OT", ascending=False)
zo["groupe"] = groupe_zone(zo.index)
print(f"Zones : {len(zo)} | OT sans zone : {df['zone'].isna().sum()}")
display(zo.head(15).round(3))

g = df.assign(groupe=groupe_zone(df["zone"])).groupby("groupe").agg(nb_OT=("codeOT", "size"),
                                                                    taux_correctifs=("CORRECT", "mean"))
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "xy"}, {"type": "domain"}]],
                    subplot_titles=["15 zones les plus chargées", "Groupes de zones"])
t = zo.head(15)
fig.add_trace(go.Bar(x=t.index, y=t["nb_OT"], marker_color="#2a78d6", showlegend=False), 1, 1)
fig.add_trace(go.Pie(labels=g.index, values=g["nb_OT"], hole=0.45), 1, 2)
fig.update_layout(title="Répartition des OT par zone", template=TEMPLATE, height=420)
fig.show()
display(g.round(3))

### Analyse & Interprétation


- **40 zones avant nettoyage, 38 après** : `"DRS "` (avec un espace) est fusionnée avec `"DRS"`, et 32 OT dont la zone n'était qu'un espace invisible deviennent « sans zone ».
- **Dakar** : 55 % des OT, taux de correctifs de **77 %**.
- **Régions** (zones `DR…`) : 30 % des OT, taux de **49 %**.
- **Autres** : 16 % des OT, dont la zone `1000` (13 489 OT, 86 % de correctifs) et `GARAGE` (véhicules, 1 %).

**Conséquence :** Dakar et les régions ne se ressemblent pas (part de correctifs, ancienneté de la saisie dans la GMAO). On évaluera donc le modèle **séparément pour Dakar et pour les régions**.

### 5.7 `coutMaintenance`

### Description de l'étape


Coût total de l'intervention. Les coûts sont très asymétriques (beaucoup de petits, quelques très gros) : le boxplot est tracé en **log(1 + coût)**.

In [ ]:
print(df["coutMaintenance"].describe(percentiles=[.5, .75, .9, .99]).round(0).to_string())
print(f"\nOT à coût nul : {(df['coutMaintenance'] == 0).mean():.1%} | correctifs à coût nul : {(corr['coutMaintenance'] == 0).mean():.1%}")
display(df.groupby("CORRECT")["coutMaintenance"].agg(["mean", "median", lambda s: (s > 0).mean()])
          .rename(index=NOMS_OT, columns={"<lambda_0>": "part avec un coût"}).round(2))
fig = boxplots_par_classe(df, ["coutMaintenance"], "CORRECT", NOMS_OT, COUL_OT,
                          "Coût de maintenance : correctifs contre autres OT", ncols=1, log=True)
fig.update_layout(height=420)
fig.show()

### Analyse & Interprétation


- **34,5 % des OT ont un coût nul**, y compris 27,7 % des correctifs : le coût n'est pas toujours renseigné.
- Les correctifs coûtent nettement plus : médiane de **45 597** contre **6 782** pour les autres OT.
- Les coûts sont très asymétriques (jusqu'à 399 millions) : d'où l'échelle logarithmique.

**Attention :** le coût d'un OT n'est connu **qu'après** l'intervention. Pour le modèle, on n'utilisera que le **coût des correctifs passés** (`log_cout12`, coût des 12 derniers mois), jamais celui de la panne à prédire.

### 5.8 `Equipement_Parent`

### Description de l'étape

L'équipement « mère ». Est-il renseigné ? Le parent est-il lui-même un équipement du fichier (sinon, on ne peut pas connaître ses pannes) ?

In [ ]:
par = df["Equipement_Parent"].dropna()
eqs = set(df["codeEquipement"])
print(f"OT avec un parent : {df['Equipement_Parent'].notna().mean():.1%}")
print(f"Parents distincts : {par.nunique():,}")
print(f"Parents qui sont aussi des équipements du fichier : {pd.Series(par.unique()).isin(eqs).mean():.1%}")
print(f"OT dont le parent est un équipement du fichier : {par.isin(eqs).mean():.1%}")

### Analyse & Interprétation


- **88 %** des OT ont un parent renseigné, avec 10 860 parents distincts.
- Mais **seuls 54 % de ces parents sont eux-mêmes des équipements du fichier**, ce qui couvre 44 % des OT. Les autres parents sont des codes de regroupement (par exemple `POSTES_SDDRCO2`) ou des lignes qui n'ont pas d'OT propres.

**Conséquence :** on ne peut connaître les pannes du parent que pour un peu moins de la moitié des OT. La variable `parent_c3` sera donc **souvent nulle**, et son apport sera limité.

### 5.9 Lien des colonnes avec la cible (au niveau de l'OT)

### Description de l'étape


- **Catégorielles** : V de Cramér avec `CORRECT` (0 = aucun lien, 1 = lien total).
- **Numériques** : information mutuelle avec `CORRECT` (0 = aucune information).

On calcule aussi ces mesures pour des colonnes écartées, pour montrer **pourquoi** un lien fort n'est pas forcément utile.

> **Ce n'est pas la sélection des variables du modèle.** Ici, on se demande si une colonne distingue un correctif d'un autre OT. Plusieurs colonnes sont remplies **pendant ou après** l'intervention : elles « connaissent » déjà la réponse. Le V de Cramér est aussi plus élevé pour les colonnes à beaucoup de modalités.

In [ ]:
cat = ["familleEquipement", "fonction", "zone", "etatOT", "classeIntervention", "entiteIntervention", "entiteAction"]
v = pd.DataFrame({"V de Cramér": [cramers_v(df[c].fillna("INCONNU").astype(str), df["CORRECT"]) for c in cat],
                  "modalités": [df[c].nunique() for c in cat],
                  "gardée": ["oui" if c in COLONNES_GARDEES else "non" for c in cat]}, index=cat).sort_values("V de Cramér")

num = ["coutMaintenance", "CoutMainDoeuvre", "CoutMateriel", "CoutService", "CoutMoyen"]
mi = pd.Series(mutual_info_classif(df[num].fillna(0), df["CORRECT"], random_state=RANDOM_STATE), index=num).sort_values()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Catégorielles : V de Cramér", "Numériques : information mutuelle"])
fig.add_trace(go.Bar(x=v["V de Cramér"], y=v.index, orientation="h", showlegend=False, text=v["V de Cramér"].round(2),
                     marker_color=np.where(v["gardée"] == "oui", "#2a78d6", "#b9b8b2")), 1, 1)
fig.add_trace(go.Bar(x=mi.values, y=mi.index, orientation="h", showlegend=False, text=mi.round(3),
                     marker_color=np.where(mi.index.isin(COLONNES_GARDEES), "#2a78d6", "#b9b8b2")), 1, 2)
fig.update_layout(title="Lien avec CORRECT (bleu = colonne gardée, gris = écartée)", template=TEMPLATE, height=420)
fig.show()
display(v.sort_values("V de Cramér", ascending=False).round(3))
print("Information mutuelle des coûts avec CORRECT :")
print(mi.sort_values(ascending=False).round(3).to_string())

In [ ]:
taux = df.groupby("familleEquipement")["CORRECT"].agg(taux="mean", nb_OT="size").sort_values("nb_OT", ascending=False)
print("Taux de correctifs des 10 familles les plus fréquentes :")
display(taux.head(10).round(2))

### Analyse & Interprétation


**Catégorielles :**
- `familleEquipement` (0,72) et `fonction` (0,66) sont les plus liées au type d'OT, mais ce sont aussi celles qui ont **le plus de modalités** (268 et 154), ce qui gonfle le V de Cramér.
- `classeIntervention` (0,64) et `entiteIntervention` (0,58) sont très liées, mais elles sont **remplies au moment de l'intervention** : elles décrivent l'intervention elle-même. Ce lien fort ne les rend pas utiles pour prédire une panne future.
- `zone` (0,45) et `etatOT` (0,35) ont un lien moyen.

**Numériques :** `coutMaintenance` apporte le plus d'information (0,14) ; ses composantes en apportent moins chacune. C'est cohérent avec le choix de garder seulement le coût total.

**Taux par famille :** de 1 % (`V-EXPLO`) à 97 % (`CABLE_HTA`). La famille distingue donc très bien les types d'activité.

**À retenir :** ces mesures confirment le choix des colonnes. La vraie sélection des variables du modèle se fait sur le panel, en phases 6 et 7.

### 5.10 Colonnes écartées

### Description de l'étape

Les colonnes non gardées et la raison de leur retrait (raisons définies dans `settings/params.py`).

In [ ]:
pd.DataFrame([{"colonne": c, "raison": r, "% manquantes": round(df[c].isna().mean() * 100, 1)}
              for c, r in COLONNES_ECARTEES.items()])

### Analyse & Interprétation


Chaque colonne écartée l'est pour une raison précise : **vide ou presque**, **répète une autre colonne**, **décrit l'intervention** plutôt que l'équipement, ou **décision de l'encadrant**.

« Écartée » ne veut pas dire inutilisable pour toujours : `classeIntervention` (part des pannes électriques) ou `dateFinPrevue` (durée des interventions passées) pourraient devenir des variables d'historique plus tard.

## 6. Feature engineering

### Description de l'étape


**Pourquoi changer de niveau.** Le fichier contient **une ligne par OT**. Le modèle doit répondre, pour un équipement et un jour donné : *« est-il à risque de panne dans les h prochains jours (10, 34, 90…) ? Dans combien de jours la prochaine panne est-elle attendue ? »*. Il faut donc un tableau où chaque ligne décrit **un équipement à une date donnée**, avec son passé (les variables) et ce qui s'est passé ensuite (la cible). **Aucune de ces variables n'existe dans le fichier brut : toutes sont créées.**

**La date de coupure.** On se place au **dernier jour de chaque mois** (31 janvier 2019, 28 février 2019…). À chaque date de coupure `d`, pour chaque équipement déjà connu :
- les **variables** résument son passé, en n'utilisant que les OT **réalisés au plus tard le jour `d` et déjà saisis le jour `d`** (règle anti-fuite) ;
- la **cible** est le **délai, en jours, jusqu'à sa prochaine panne**.

**Pourquoi une cible « délai » (approche de survie).** Avec un délai, un seul modèle peut répondre pour **n'importe quel horizon** : la probabilité de panne dans 10 jours, dans 34 jours ou dans 90 jours se lit sur la même courbe. Une cible « panne le mois suivant » ne permettrait de répondre que pour un mois.

**La censure.** On ne suit chaque date de coupure que pendant une durée limitée : au plus 365 jours (`H_MAX`), sans dépasser la fin d'observation de l'ensemble ni le trou d'extraction de juin-juillet 2025. Si aucune panne n'arrive pendant ce suivi, on sait seulement que l'équipement a tenu **au moins** ce nombre de jours : on dit que l'observation est **censurée**.

| Colonne de la cible | Signification |
|---|---|
| `duree` | jours entre la date de coupure et la prochaine panne, ou durée du suivi si aucune panne |
| `evenement` | 1 = une panne a été observée ; 0 = pas de panne pendant le suivi (censure) |

**Les variables créées** (fonction `variables_a_date`, fichier `src/features/build.py`) :

| Variable | Signification | Si l'information n'existe pas |
|---|---|---|
| `n_corr_7j`, `n_corr_30j`, `n_corr_90j`, `n_corr_365j` | pannes sur les 7, 30, 90, 365 derniers jours | 0 |
| `n_corr_total` | pannes depuis 2019 | 0 |
| `n_prev_365j`, `n_prev_total` | préventifs sur 365 jours, depuis 2019 | 0 |
| `jours_depuis_corr`, `jours_depuis_prev` | jours depuis la dernière panne, le dernier préventif | 3 000 = jamais |
| `ecart_moyen_corr` | écart moyen, en jours, entre deux pannes de l'équipement (son « rythme ») | 3 000 si moins de 2 pannes |
| `log_cout_365j` | coût des pannes sur 365 jours, en log(1 + coût) | 0 |
| `parent_corr_90j` | pannes de l'équipement parent sur 90 jours | 0 |
| `anciennete_jours` | jours depuis le premier OT connu de l'équipement (ancienneté dans la GMAO) | toujours connu |
| `mois` | mois de la date de coupure (saison) | toujours connu |
| `familleEquipement`, `fonction`, `zone` | dernière valeur connue le jour de la coupure | catégorie manquante |

Toutes les analyses de cette phase et de la suivante portent sur **l'entraînement** (dates de coupure de janvier 2019 à octobre 2024) et sans les codes hors périmètre : on ne regarde jamais la validation ni le test.

In [ ]:
ot = prepare(data)
ensembles = construire_ensembles(ot)
tr = ensembles["train"]
tr = tr[~tr["hors_perimetre"]].copy()
H = HORIZON_REFERENCE
tr["panne_30j"] = etiquette_horizon(tr, H)          # panne dans les 30 jours : 1 / 0 (inconnu si suivi trop court)
connue = tr[tr["panne_30j"].notna()]                 # lignes dont on sait s'il y a eu panne dans les 30 jours
NOMS_30 = {1: "Panne dans les 30 jours", 0: "Pas de panne dans les 30 jours"}
logger.info(f"Entraînement, équipements du réseau : {len(tr):,} lignes, {tr[EQ].nunique():,} équipements, "
            f"{tr['date_coupure'].nunique()} dates de coupure")
display(tr[[EQ, "date_coupure"] + FEATURES_NUM + FEATURES_CAT + [DUREE, EVENEMENT]].sample(5, random_state=RANDOM_STATE))

### Analyse & Interprétation


- Après préparation : **164 222 OT** sur **43 840 équipements**, réalisés du 1er janvier 2019 au 31 décembre 2025.
- **Entraînement** : 957 924 lignes (70 dates de coupure, de janvier 2019 à octobre 2024). **Validation** : 215 683 lignes (6 dates). **Test** : 168 425 lignes (4 dates, d'août à novembre 2025). **Futur** : 43 805 équipements au 31 décembre 2025, pour la prévision réelle.
- Sans les codes hors périmètre, l'entraînement compte **795 975 lignes** sur **25 193 équipements du réseau** : c'est sur ce tableau que portent les analyses qui suivent.

Chaque ligne se lit ainsi : « à la date de coupure, voici ce qu'on savait de l'équipement ; `duree` et `evenement` disent dans combien de jours sa prochaine panne est arrivée, ou combien de jours il a été suivi sans panne ».

### 6.1 Exemple : un équipement suivi dans le temps

### Description de l'étape


Pour comprendre le tableau, on suit **un seul équipement** : ses pannes réelles, puis ses lignes à plusieurs dates de coupure. On vérifie que les variables et la cible correspondent bien à son histoire.

In [ ]:
exemple = (tr[tr["n_corr_total"].between(5, 15)].groupby(EQ).size().sort_values(ascending=False).index[0])
pannes_ex = ot[(ot[EQ] == exemple) & (ot["typeIntervention"] == TYPE_PANNE)]
print(f"Équipement : {exemple} ({tr.loc[tr[EQ] == exemple, 'familleEquipement'].iloc[-1]})")
print("Dates de ses pannes (réalisation) :", ", ".join(pannes_ex["date_realisation"].dt.strftime("%d/%m/%Y").tolist()))
lignes_ex = tr[tr[EQ] == exemple].sort_values("date_coupure")
display(lignes_ex[["date_coupure", "n_corr_30j", "n_corr_365j", "n_corr_total", "jours_depuis_corr",
                   "ecart_moyen_corr", DUREE, EVENEMENT]].tail(8))

### Analyse & Interprétation


L'équipement `LM0403TP_HAN-KALIA` (un câble HTA) a eu **5 pannes entre le 22 janvier et le 28 février 2019**, puis **plus rien pendant 5 ans**, puis une panne le **25 avril 2024**.

- **Au 31 mars 2024** : 5 pannes connues, la dernière il y a 1 858 jours. Son écart moyen entre pannes est de 9,25 jours (5 pannes en 37 jours). La prochaine panne arrive **25 jours plus tard** (le 25 avril) : `duree = 25`, `evenement = 1`.
- **Au 30 avril 2024** : la panne du 25 avril n'est pas encore comptée, car cette ligne a été construite à partir de ce qu'on savait le 30 avril, et elle avait peut-être été saisie plus tard. Aucune nouvelle panne n'arrive avant la fin d'observation de l'entraînement (30 novembre 2024) : la ligne est **censurée** après 214 jours (`evenement = 0`).
- **Du 31 mai au 31 octobre 2024** : la panne d'avril est visible (`n_corr_365j = 1`, `jours_depuis_corr` augmente de 30 jours par mois) et l'écart moyen passe à 384 jours. Le suivi se raccourcit à mesure qu'on approche de la fin d'observation (183, 153… 30 jours).

Cet exemple montre les trois idées clés : les variables ne voient que le passé **déjà saisi**, la cible est un **délai**, et une ligne sans panne observée est **censurée**, pas comptée comme « jamais de panne ».

### 6.2 La cible : délai jusqu'à la prochaine panne

### Description de l'étape


- Part des lignes avec une panne observée et part des lignes censurées.
- Répartition des délais quand une panne est observée.
- **Courbe de Kaplan-Meier** : probabilité de panne dans les h jours, pour **tout h** entre 1 et 365. Elle tient compte de la censure : une ligne suivie 60 jours sans panne compte comme « sans panne » jusqu'au jour 60, puis n'est plus comptée.

In [ ]:
print(f"Panne observée : {tr[EVENEMENT].mean():.1%} des lignes | censurées : {1 - tr[EVENEMENT].mean():.1%}")
print("Délai (jours) quand une panne est observée :")
print(tr.loc[tr[EVENEMENT] == 1, DUREE].describe(percentiles=[.25, .5, .75]).round(0).to_string())

km = kaplan_meier(tr[DUREE], tr[EVENEMENT])
table = pd.DataFrame({"horizon (jours)": [1, 7, 10, 30, 34, 90, 180, 365]})
table["probabilité de panne"] = [f"{proba_panne(km, h):.2%}" for h in table["horizon (jours)"]]
display(table)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Délai jusqu'à la panne (pannes observées)",
                                                    "Probabilité de panne dans les h jours (Kaplan-Meier)"])
h_, b_ = np.histogram(tr.loc[tr[EVENEMENT] == 1, DUREE], bins=np.arange(0, 372, 7))
fig.add_trace(go.Bar(x=b_[:-1], y=h_, marker_color=COUL_Y[1], showlegend=False), 1, 1)
fig.add_trace(go.Scatter(x=km["jour"], y=1 - km["survie"], line_color=COUL_Y[1], showlegend=False), 1, 2)
fig.update_xaxes(title_text="jours", row=1, col=1); fig.update_xaxes(title_text="horizon h (jours)", row=1, col=2)
fig.update_yaxes(title_text="nombre de pannes", row=1, col=1); fig.update_yaxes(tickformat=".0%", row=1, col=2)
fig.update_layout(template=TEMPLATE, height=400)
fig.show()

### Analyse & Interprétation


- **Une panne est observée pour 21,7 % des lignes** ; les 78,3 % restantes sont **censurées** (pas de panne pendant le suivi, qui dure au plus 365 jours).
- Quand une panne arrive, elle arrive en moyenne **137 jours** après la date de coupure (médiane 116 jours) ; un quart des pannes arrive dans les 47 jours.

**Probabilité de panne selon l'horizon** (courbe de Kaplan-Meier, tous équipements du réseau) :

| Horizon | 1 j | 7 j | 10 j | 30 j | 34 j | 90 j | 180 j | 365 j |
|---|---|---|---|---|---|---|---|---|
| Probabilité de panne | 0,14 % | 0,99 % | 1,39 % | 3,76 % | 4,18 % | 9,33 % | 15,7 % | 25,0 % |

**À retenir :**
- La courbe permet de lire le risque pour **n'importe quel horizon**, par exemple 10 ou 34 jours, ce que demandait le maître de stage. À 30 jours, on retrouve le taux de l'ancienne version mensuelle (3,8 %).
- **Les pannes sont rares à court terme** : 0,14 % par jour en moyenne. Prédire une panne pour demain sera beaucoup plus difficile que pour les 30 ou 90 prochains jours.
- **Pour un équipement « moyen », la probabilité de panne dans l'année reste inférieure à 50 %** (25 %) : son délai médian dépasse donc 365 jours. Le « quand » ne pourra être estimé précisément que pour les équipements à risque élevé (voir 6.3 et 6.4).

### 6.3 Historique des pannes

### Description de l'étape


- **Boxplots** : chaque variable comparée entre les lignes suivies d'une panne dans les 30 jours (rouge) et les autres (bleu). Échelle log(1 + valeur), car la plupart des équipements n'ont aucune panne récente.
- **Courbes de survie par groupe** : probabilité de panne dans les h jours selon le nombre de pannes sur 365 jours.

In [ ]:
V = ["n_corr_7j", "n_corr_30j", "n_corr_90j", "n_corr_365j", "n_corr_total"]
display(connue.groupby("panne_30j")[V].mean().rename(index=NOMS_30).round(2))
boxplots_par_classe(connue, V, "panne_30j", NOMS_30, COUL_Y, "Historique des pannes selon la panne dans les 30 jours", log=True).show()
groupes = pd.cut(tr["n_corr_365j"], [-1, 0, 1, 2, 4, 9, np.inf], labels=["0", "1", "2", "3-4", "5-9", "10 et +"])
fig, resume_km = courbes_survie(tr, groupes, "Probabilité de panne selon le nombre de pannes sur 365 jours", ordre=list(groupes.cat.categories), horizons=HORIZONS)
fig.show()
display(resume_km)

### Analyse & Interprétation


- **Avant une panne dans les 30 jours, l'équipement a nettement plus de pannes passées** : 2,37 pannes sur 365 jours contre 0,70, et 5,07 pannes au total contre 1,94. Les boîtes rouges sont plus hautes pour toutes les variables, sauf `n_corr_7j`, écrasée à 0, car une panne dans la dernière semaine est rare.
- **Les courbes de survie s'écartent nettement** selon le nombre de pannes sur 365 jours :

| Pannes sur 365 j | P(panne ≤ 7 j) | P(panne ≤ 30 j) | P(panne ≤ 90 j) | Délai médian |
|---|---|---|---|---|
| 0 | 0,4 % | 1,7 % | 4,7 % | plus d'un an |
| 1 | 1,0 % | 3,9 % | 10,0 % | plus d'un an |
| 2 | 1,9 % | 7,5 % | 18,7 % | plus d'un an |
| 3-4 | 3,5 % | 13,0 % | 29,0 % | 232 jours |
| 5-9 | 6,1 % | 20,9 % | 43,1 % | 120 jours |
| 10 et + | 11,0 % | 30,8 % | 55,6 % | **71 jours** |

**À retenir :** l'historique est le **signal le plus fort**. Plus un équipement a eu de pannes dans l'année, plus la suivante arrive **tôt**. Pour les équipements très touchés, on peut déjà annoncer un délai probable (environ 2 à 4 mois).

### 6.4 Récence et rythme des pannes

### Description de l'étape


- `jours_depuis_corr` : jours depuis la dernière panne (3 000 = jamais).
- `ecart_moyen_corr` : écart moyen entre deux pannes de l'équipement (3 000 si moins de deux pannes).

In [ ]:
V = ["jours_depuis_corr", "ecart_moyen_corr"]
print(f"Jamais de panne connue : {(tr['jours_depuis_corr'] == JOURS_JAMAIS).mean():.1%} | "
      f"moins de deux pannes : {(tr['ecart_moyen_corr'] == JOURS_JAMAIS).mean():.1%}")
display(connue.groupby("panne_30j")[V].median().rename(index=NOMS_30).round(0))
boxplots_par_classe(connue, V, "panne_30j", NOMS_30, COUL_Y, "Récence et rythme selon la panne dans les 30 jours", ncols=2).show()
groupes = pd.cut(tr["jours_depuis_corr"], [-1, 7, 30, 90, 365, JOURS_JAMAIS - 1, JOURS_JAMAIS],
                 labels=["0-7 j", "8-30 j", "31-90 j", "91-365 j", "plus d'un an", "jamais"])
fig, resume_km = courbes_survie(tr, groupes, "Probabilité de panne selon le temps écoulé depuis la dernière panne", ordre=list(groupes.cat.categories), horizons=HORIZONS)
fig.show()
display(resume_km)

### Analyse & Interprétation


- **19,2 %** des lignes n'ont jamais eu de panne connue ; **63,9 %** en ont moins de deux, donc pas d'écart moyen calculable (valeur 3 000).
- Médianes : la dernière panne date de **139 jours** avant une panne dans les 30 jours, contre **524 jours** sinon ; l'écart moyen entre pannes est de **194 jours** pour les équipements qui vont tomber en panne.
- **Le risque baisse régulièrement avec le temps écoulé depuis la dernière panne :**

| Dernière panne | P(panne ≤ 7 j) | P(panne ≤ 30 j) | P(panne ≤ 90 j) | Délai médian |
|---|---|---|---|---|
| 0-7 jours | 6,5 % | 16,9 % | 29,7 % | 309 jours |
| 8-30 jours | 3,7 % | 12,7 % | 25,8 % | 352 jours |
| 31-90 jours | 2,3 % | 8,7 % | 20,1 % | plus d'un an |
| 91-365 jours | 1,2 % | 4,7 % | 12,3 % | plus d'un an |
| plus d'un an | 0,4 % | 1,8 % | 5,1 % | plus d'un an |
| jamais | 0,3 % | 1,4 % | 3,8 % | plus d'un an |

**À retenir :** une panne récente annonce une nouvelle panne : le risque à 7 jours est **plus de 20 fois** plus élevé dans la semaine qui suit une panne que pour un équipement jamais tombé en panne. Les pannes arrivent **en série** (réparation incomplète, équipement fragile). L'**écart moyen** apporte une information nouvelle, utile pour le « quand » : le rythme habituel des pannes de l'équipement.

### 6.5 Préventifs

### Description de l'étape

Nombre de préventifs et temps écoulé depuis le dernier préventif.

In [ ]:
V = ["n_prev_365j", "n_prev_total", "jours_depuis_prev"]
print(f"Lignes avec au moins un préventif sur 365 jours : {(tr['n_prev_365j'] > 0).mean():.1%}")
boxplots_par_classe(connue, V[:2], "panne_30j", NOMS_30, COUL_Y, "Préventifs selon la panne dans les 30 jours", ncols=2, log=True).show()
groupes = pd.Series(np.where(tr["n_prev_365j"] > 0, "au moins un préventif sur 365 j", "aucun préventif sur 365 j"), index=tr.index)
fig, resume_km = courbes_survie(tr, groupes, "Probabilité de panne selon la présence de préventifs", horizons=HORIZONS)
fig.show()
display(resume_km)

### Analyse & Interprétation


- Seules **20,4 %** des lignes ont eu au moins un préventif sur 365 jours.
- Les boîtes se ressemblent : le signal est **faible**.
- Le risque est **plus élevé** avec préventif (5,3 % de panne dans les 30 jours) que sans (3,4 %).

**À retenir :** comme dans la version mensuelle, le préventif ne montre pas d'effet protecteur dans ces données. Il est réalisé sur les **équipements importants ou fragiles**, qui tombent aussi le plus en panne : il renseigne surtout sur le **type d'équipement**. Cela ne prouve pas que le préventif soit inutile, car on ne sait pas combien de pannes il a évitées.

### 6.6 Coût, parent, ancienneté et saison

### Description de l'étape


`log_cout_365j`, `parent_corr_90j` et `anciennete_jours` en boxplots ; `mois` en probabilité de panne dans les 30 jours selon le mois de la date de coupure.

In [ ]:
V = ["log_cout_365j", "parent_corr_90j", "anciennete_jours"]
display(connue.groupby("panne_30j")[V].mean().rename(index=NOMS_30).round(2))
print(f"Lignes dont le parent a eu au moins une panne sur 90 jours : {(tr['parent_corr_90j'] > 0).mean():.1%}")
boxplots_par_classe(connue, V, "panne_30j", NOMS_30, COUL_Y, "Coût, parent et ancienneté selon la panne dans les 30 jours").show()
s = connue.groupby("mois")["panne_30j"].mean()
fig = px.bar(x=s.index, y=s.values, template=TEMPLATE, labels={"x": "mois de la date de coupure", "y": "panne dans les 30 jours"},
             title="Probabilité de panne dans les 30 jours selon le mois")
fig.update_traces(marker_color=COUL_Y[1]); fig.update_layout(height=350, yaxis_tickformat=".1%")
fig.show()
print("Par mois :", s.round(4).to_dict())

### Analyse & Interprétation


- **`log_cout_365j`** : nettement plus élevé avant une panne (8,64 contre 4,41). Le coût résume le nombre et la **gravité** des pannes de l'année.
- **`parent_corr_90j`** : seules 8,5 % des lignes ont un parent avec une panne récente, et la moyenne est même plus faible avant une panne (0,08 contre 0,14). Signal faible et peu fiable, car le parent n'est connu que pour une partie des équipements (voir 5.8).
- **`anciennete_jours`** : identique dans les deux groupes (722 contre 731 jours). L'ancienneté dans la GMAO ne distingue pas les pannes.
- **`mois`** : la probabilité de panne dans les 30 jours varie peu, de 3,4 % (fin mars) à 4,2 % (fin juillet). Léger pic en juin-juillet (peut-être le début de l'hivernage, à confirmer), mais l'effet est faible.

### 6.7 Catégories : famille, fonction, zone

### Description de l'étape


Probabilité de panne dans les 30 jours pour les modalités les plus fréquentes, et information mutuelle avec la panne dans les 30 jours.

In [ ]:
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.2,
                    subplot_titles=["familleEquipement (15 plus fréquentes)", "fonction (15 plus fréquentes)", "zone (10 plus fréquentes)"])
for i, (c, n) in enumerate([("familleEquipement", 15), ("fonction", 15), ("zone", 10)], start=1):
    t = connue.groupby(c, observed=True)["panne_30j"].agg(taux="mean", lignes="size").sort_values("lignes", ascending=False).head(n)
    t = t.sort_values("taux")
    fig.add_trace(go.Bar(x=t["taux"], y=t.index.astype(str), orientation="h", marker_color=COUL_Y[1], showlegend=False), 1, i)
fig.update_layout(title="Probabilité de panne dans les 30 jours par modalité (entraînement)", template=TEMPLATE, height=520)
fig.update_xaxes(tickformat=".0%")
fig.show()

ech = connue.sample(TAILLE_ECHANTILLON_GRAPHIQUES, random_state=RANDOM_STATE)
mi = pd.Series(mutual_info_classif(ech[FEATURES_CAT].apply(lambda x: x.cat.codes + 1), ech["panne_30j"],
                                   discrete_features=True, random_state=RANDOM_STATE), index=FEATURES_CAT)
print("Information mutuelle avec la panne dans les 30 jours :", mi.round(4).sort_values(ascending=False).to_dict())
print("Modalités présentes :", {c: tr[c].nunique() for c in FEATURES_CAT})

### Analyse & Interprétation


- La **famille** est la catégorie la plus informative (information mutuelle 0,012), devant la **fonction** (0,009). La **zone** apporte très peu (0,001), car la fonction (l'unité) dit déjà où se trouve l'équipement.
- Les probabilités de panne dans les 30 jours varient fortement d'une famille à l'autre : de **0,3 %** (`RELAY-HTA`) à **15,4 %** (`ITI`), en passant par 5,4 % pour `CABLE_HTA` et 5,0 % pour `DEPART_BT`. Elles varient aussi d'une unité à l'autre.
- Dans l'entraînement, il reste 239 familles, 89 fonctions et 18 zones.

**À retenir :** famille et fonction sont à garder ; la zone sera sans doute peu utile au modèle, mais reste indispensable pour évaluer séparément Dakar et les régions.

### 6.8 Résumé des variables numériques

### Description de l'étape


- **Variance nulle** : une variable qui a toujours la même valeur n'apporte rien.
- **Corrélation de Spearman** de chaque variable avec la panne dans les 30 jours, et avec la panne dans les 90 jours pour vérifier que le classement est le même à un autre horizon.

In [ ]:
var_nulle = [c for c in FEATURES_NUM if tr[c].std() == 0]
print("Variables à variance nulle :", var_nulle or "aucune")
corr_30 = correlation_cible(connue, FEATURES_NUM, "panne_30j")
tr["panne_90j"] = etiquette_horizon(tr, 90)
corr_90 = correlation_cible(tr[tr["panne_90j"].notna()], FEATURES_NUM, "panne_90j")
display(pd.DataFrame({"Spearman (30 jours)": corr_30, "Spearman (90 jours)": corr_90}).round(3))
fig = px.bar(x=corr_30.values, y=corr_30.index, orientation="h", color=corr_30.values, template=TEMPLATE,
             color_continuous_scale="RdBu_r", range_color=[-0.3, 0.3],
             title="Corrélation de Spearman de chaque variable avec la panne dans les 30 jours (entraînement)")
fig.add_vline(x=0, line_dash="dash")
fig.update_layout(height=480, yaxis={"categoryorder": "total ascending"}, coloraxis_showscale=False,
                  xaxis_title="corrélation", yaxis_title=None)
fig.show()

### Analyse & Interprétation


- **Aucune variable n'a une variance nulle.**
- **Signal fort** (|corrélation| autour de 0,12 à 0,17 à 30 jours) : `n_corr_365j` (0,17), `n_corr_90j`, `n_corr_total`, `jours_depuis_corr` (-0,15), `ecart_moyen_corr` (-0,14), `log_cout_365j` (0,14), `n_corr_30j`.
- **Signal faible** : préventifs, `n_corr_7j`, `parent_corr_90j`.
- **Pas de signal** : `anciennete_jours` et `mois`.
- **À 90 jours, le classement est le même et les corrélations sont plus fortes** (`n_corr_365j` : 0,24, `n_corr_total` : 0,23) : les variables d'historique prédisent mieux à moyen terme qu'à très court terme.

## 7. Corrélations croisées

### Description de l'étape


**Heatmap** des corrélations de Spearman entre toutes les variables numériques (entraînement, sur un échantillon de 200 000 lignes).

**Règle de redondance :** si deux variables ont une corrélation supérieure au seuil en valeur absolue (`SEUIL_CORRELATION` = 0,8), elles apportent presque la même information : on garde celle qui est **la plus liée à la panne dans les 30 jours** et on retire l'autre. Comme dans le projet modèle, cette règle définit la liste des variables pour les **modèles linéaires** (`FEATURES_LINEAR`) ; les modèles à arbres utilisent toutes les variables (`FEATURES_ALL`), car ils supportent les variables redondantes.

In [ ]:
ech = connue.sample(TAILLE_ECHANTILLON_GRAPHIQUES, random_state=RANDOM_STATE)
mat = ech[FEATURES_NUM + ["panne_30j"]].rank().corr()
fig = px.imshow(mat, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1, template=TEMPLATE,
                title="Corrélations de Spearman entre variables (entraînement, panne dans les 30 jours incluse)")
fig.update_layout(height=750)
fig.show()

In [ ]:
paires = paires_redondantes(mat.drop(index="panne_30j", columns="panne_30j"), SEUIL_CORRELATION)
decisions, retirees = regle_redondance(paires, corr_30)
display(decisions.round(3))
FEATURES_LINEAR_PROPOSEES = [c for c in FEATURES_NUM if c not in retirees] + FEATURES_CAT
print("Variables retirées pour les modèles linéaires :", retirees)
print("FEATURES_LINEAR =", FEATURES_LINEAR_PROPOSEES)
print("Identique à la liste enregistrée dans settings/params.py :", FEATURES_LINEAR_PROPOSEES == FEATURES_LINEAR)

### Analyse & Interprétation


**Heatmap :** comme dans la version mensuelle, deux blocs : l'**historique des pannes** (`n_corr_*`, `log_cout_365j`, et en négatif `jours_depuis_corr` et `ecart_moyen_corr`) et les **préventifs** (`n_prev_*`, et en négatif `jours_depuis_prev`). Les autres variables sont presque indépendantes.

**Cinq paires dépassent 0,8 en valeur absolue :**

| Paire | Corrélation | Gardée | Retirée (modèles linéaires) |
|---|---|---|---|
| `n_prev_total` – `jours_depuis_prev` | -0,96 | `jours_depuis_prev` | `n_prev_total` |
| `n_corr_365j` – `log_cout_365j` | 0,89 | `n_corr_365j` | `log_cout_365j` |
| `n_corr_total` – `ecart_moyen_corr` | -0,86 | `n_corr_total` | `ecart_moyen_corr` |
| `n_corr_365j` – `jours_depuis_corr` | -0,85 | `n_corr_365j` | `jours_depuis_corr` |
| `n_prev_365j` – `jours_depuis_prev` | -0,84 | `jours_depuis_prev` | `n_prev_365j` |

**Décision (comme dans le projet modèle) :**
- `FEATURES_ALL` (17 variables) pour les **modèles à arbres** (LightGBM, forêts de survie), qui supportent les variables redondantes et peuvent tirer parti de `jours_depuis_corr` et `ecart_moyen_corr`, importants pour estimer **quand** la panne arrivera ;
- `FEATURES_LINEAR` (12 variables, sans les 5 retirées) pour les **modèles linéaires** (régression logistique, modèle de Cox).

Les deux listes sont enregistrées dans `settings/params.py`.

## 8. Synthèse et décisions

### 8.1 Constats clés

| # | Constat | Conséquence pour le projet |
|---|---|---|
| 1 | Les dates sont stockées en texte ; quelques dates aberrantes (1970, après 2026) | conversion des dates, période limitée à 2019-2025 |
| 2 | Saisies en masse en juillet et septembre 2024 ; 10 % des correctifs saisis plus de 3 mois après | panne **datée par la réalisation**, le modèle ne voit que les OT **déjà saisis** |
| 3 | Extraction incomplète en juin et juillet 2025 | aucune date de coupure ni aucun suivi dans ce trou |
| 4 | 732 OT en double (ligne `DD`) | une ligne par OT, par précaution |
| 5 | Localités, unités, groupes d'équipements et véhicules dans le fichier | **hors périmètre** : exclus de l'analyse, de l'évaluation et de la liste de risque |
| 6 | Le maître de stage veut un horizon libre (10 jours, 34 jours…) et la date probable de la panne | **approche de survie** : cible = délai jusqu'à la prochaine panne, avec censure |
| 7 | Risque faible à court terme (1 % à 7 jours, 3,8 % à 30 jours, 25 % à 365 jours) | évaluer à plusieurs horizons ; le « quand » sera fiable surtout pour les équipements à risque élevé |
| 8 | Les pannes se répètent et arrivent en série | historique, récence et rythme des pannes = meilleurs signaux |
| 9 | Dakar et les régions ont des profils différents | évaluation séparée |
| 10 | Le parent n'est un équipement du fichier que pour 44 % des OT | `parent_corr_90j` peu fiable |

### 8.2 Découpage temporel

| Ensemble | Dates de coupure | Fin d'observation | Rôle |
|---|---|---|---|
| Entraînement | fin janvier 2019 à fin octobre 2024 (70 dates) | 30 novembre 2024 | le modèle apprend |
| Validation | fin novembre 2024 à fin avril 2025 (6 dates) | 31 mai 2025 | choix des variables et des réglages |
| Test | fin août à fin novembre 2025 (4 dates) | 31 décembre 2025 | note finale, utilisé une seule fois |
| Futur | 31 décembre 2025 | — | prévision réelle (janvier 2026 et au-delà) |

La fin d'observation empêche l'entraînement de « voir » les pannes de la période de validation ou de test.

### 8.3 Variables pour la modélisation

- `FEATURES_ALL` : les 14 variables numériques et les 3 catégorielles, pour les modèles à arbres.
- `FEATURES_LINEAR` : sans `log_cout_365j`, `jours_depuis_corr`, `ecart_moyen_corr`, `n_prev_365j` et `n_prev_total`, pour les modèles linéaires.
- **À vérifier en modélisation** : l'intérêt de `anciennete_jours`, `mois`, `zone` et `parent_corr_90j` (signal quasi nul).

### 8.4 Questions ouvertes pour l'encadrant

- Les codes `C4`, `401MD` et `GE401` (zone `1000`) sont-ils des équipements du réseau ?
- Les parents absents du fichier (par exemple `POSTES_SDDRCO2`) correspondent-ils à des regroupements ?
- Quels horizons comptent le plus pour les équipes (7, 30, 90 jours) ? Prédire à 1 ou 2 jours sera peu fiable.

### 8.5 Prochaine étape

`01_02_splitting.ipynb` : préparation, construction des variables et de la cible, découpage temporel avec les fonctions de `src/`, contrôles anti-fuite, puis **sauvegarde** de l'entraînement, de la validation, du test et du futur dans `data/processed/`.